Imports

In [1]:
import requests
import pandas as pd
import os
import json

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

Configuration

In [2]:
RXNORM_BASE = "https://rxnav.nlm.nih.gov/REST"

OUTPUT_FOLDER = "rxnorm_repository"

Create Output Folder

In [3]:
if not os.path.exists(
    OUTPUT_FOLDER
):

    os.makedirs(
        OUTPUT_FOLDER
    )

print(
    os.path.abspath(
        OUTPUT_FOLDER
    )
)

C:\Users\barry.peterson\OneDrive - Integra Connect LLC\Documents\Pharmacy Reports\Master Drug File\rxnorm_repository


RxNorm Client

In [4]:
class RxNormClient:

    BASE_URL = RXNORM_BASE

    def get_rxcui(
        self,
        drug_name
    ):

        try:

            response = requests.get(

                f"{self.BASE_URL}/rxcui.json",

                params={
                    "name": drug_name
                },

                timeout=30
            )

            response.raise_for_status()

            return (

                response.json()

                .get(
                    "idGroup",
                    {}
                )

                .get(
                    "rxnormId",
                    []
                )
            )

        except:

            return []

    def get_properties(
        self,
        rxcui
    ):

        try:

            response = requests.get(

                f"{self.BASE_URL}/rxcui/{rxcui}/properties.json",

                timeout=30
            )

            response.raise_for_status()

            return (

                response.json()

                .get(
                    "properties",
                    {}
                )
            )

        except:

            return {}

Build Properties Table

In [6]:
def build_properties_df(
    rxcui_list
):

    rows = []

    rx = RxNormClient()

    for rxcui in rxcui_list:

        properties = rx.get_properties(
            rxcui
        )

        if properties:

            rows.append(
                properties
            )

    return pd.DataFrame(rows)

In [7]:
properties_df = build_properties_df(
    rxcuis
)

display(
    properties_df
)

,rxcui,name,synonym,tty,language,suppress,umlscui
0,1597881,Opdivo,,BN,ENG,N,


# RxNorm Repository Builder

Load Repository

In [9]:
repository_df = pd.read_parquet(
    "repository/repository.parquet"
)

print(
    f"Repository Rows: "
    f"{len(repository_df):,}"
)


Repository Rows: 26,102


Extract Generic Names

In [10]:
generic_names_df = (

    repository_df[
        ["generic_name"]
    ]

    .dropna()

    .drop_duplicates()

    .sort_values(
        "generic_name"
    )

    .reset_index(
        drop=True
    )
)

print(
    f"Unique Generic Names: "
    f"{len(generic_names_df):,}"
)

display(
    generic_names_df.head(20)
)

Unique Generic Names: 4,208


,generic_name
0,(S)-Alpha-Methylphenethylamine Sulphate
1,"(To Deliver) Avobenzone 2%, Octisalate 4.5%, Octocrylene 7%"
2,"(To Deliver) Avobenzone 3%, Homosalate 10%, Octisalate 5%, Octocrylene 5%"
3,"(To Deliver) Avobenzone 3%, Homosalate 5%, Octisalate 10%, Octocrylene 5%"
4,0.13% Benzalkonium Chloride
5,0.13% benzalkonium chloride solution
6,0.63% Stannous Fluoride Concentrate Rinse
7,1.1% Sodium Fluoride with 5% Potassium Nitrate Toothpaste
8,1168 Burn Spray
9,1169 Hydrogen Peroxide Spray


Build RxCUI Lookup

In [11]:
def build_rxcui_lookup_df(
    generic_names_df
):

    rows = []

    rx = RxNormClient()

    total = len(
        generic_names_df
    )

    for index, generic_name in enumerate(

        generic_names_df[
            "generic_name"
        ],

        start=1
    ):

        if index % 100 == 0:

            print(

                f"{index:,}"
                f" / "
                f"{total:,}"
            )

        rxcuis = rx.get_rxcui(
            generic_name
        )

        if len(rxcuis) == 0:

            rows.append({

                "generic_name":
                    generic_name,

                "rxcui":
                    None
            })

        else:

            for rxcui in rxcuis:

                rows.append({

                    "generic_name":
                        generic_name,

                    "rxcui":
                        rxcui
                })

    return pd.DataFrame(
        rows
    )

Build Lookup Table

In [12]:
rxcui_lookup_df = (
    build_rxcui_lookup_df(
        generic_names_df
    )
)

print(
    f"Rows: "
    f"{len(rxcui_lookup_df):,}"
)

display(
    rxcui_lookup_df.head(25)
)

100 / 4,208
200 / 4,208
300 / 4,208
400 / 4,208
500 / 4,208
600 / 4,208
700 / 4,208
800 / 4,208
900 / 4,208
1,000 / 4,208
1,100 / 4,208
1,200 / 4,208
1,300 / 4,208
1,400 / 4,208
1,500 / 4,208
1,600 / 4,208
1,700 / 4,208
1,800 / 4,208
1,900 / 4,208
2,000 / 4,208
2,100 / 4,208
2,200 / 4,208
2,300 / 4,208
2,400 / 4,208
2,500 / 4,208
2,600 / 4,208
2,700 / 4,208
2,800 / 4,208
2,900 / 4,208
3,000 / 4,208
3,100 / 4,208
3,200 / 4,208
3,300 / 4,208
3,400 / 4,208
3,500 / 4,208
3,600 / 4,208
3,700 / 4,208
3,800 / 4,208
3,900 / 4,208
4,000 / 4,208
4,100 / 4,208
4,200 / 4,208
Rows: 4,222


,generic_name,rxcui
0,(S)-Alpha-Methylphenethylamine Sulphate,None
1,"(To Deliver) Avobenzone 2%, Octisalate 4.5%, Octocrylene 7%",None
2,"(To Deliver) Avobenzone 3%, Homosalate 10%, Octisalate 5%, Octocrylene 5%",None
3,"(To Deliver) Avobenzone 3%, Homosalate 5%, Octisalate 10%, Octocrylene 5%",None
4,0.13% Benzalkonium Chloride,None
5,0.13% benzalkonium chloride solution,None
6,0.63% Stannous Fluoride Concentrate Rinse,None
7,1.1% Sodium Fluoride with 5% Potassium Nitrate Toothpaste,None
8,1168 Burn Spray,None
9,1169 Hydrogen Peroxide Spray,None


RxNorm Properties

In [13]:
def build_rxnorm_properties_df(
    rxcui_lookup_df
):

    rx = RxNormClient()

    rows = []

    unique_rxcuis = (

        rxcui_lookup_df[
            "rxcui"
        ]

        .dropna()

        .unique()
    )

    total = len(
        unique_rxcuis
    )

    for index, rxcui in enumerate(

        unique_rxcuis,

        start=1
    ):

        if index % 100 == 0:

            print(

                f"{index:,}"
                f" / "
                f"{total:,}"
            )

        properties = (
            rx.get_properties(
                rxcui
            )
        )

        if properties:

            rows.append(
                properties
            )

    return pd.DataFrame(
        rows
    )

Properties Asset

In [14]:
rxnorm_properties_df = (
    build_rxnorm_properties_df(
        rxcui_lookup_df
    )
)

print(
    f"RxNorm Properties: "
    f"{len(rxnorm_properties_df):,}"
)

display(
    rxnorm_properties_df.head(20)
)

100 / 1,459
200 / 1,459
300 / 1,459
400 / 1,459
500 / 1,459
600 / 1,459
700 / 1,459
800 / 1,459
900 / 1,459
1,000 / 1,459
1,100 / 1,459
1,200 / 1,459
1,300 / 1,459
1,400 / 1,459
RxNorm Properties: 1,457


,rxcui,name,synonym,tty,language,suppress,umlscui
0,1427084,7-oxo-prasterone,7-keto-dehydroepiandrosterone,IN,ENG,N,
1,1921069,abaloparatide,,IN,ENG,N,
2,161,acetaminophen,,IN,ENG,N,
3,167,acetazolamide,,IN,ENG,N,
4,1372250,Achillea millefolium extract,Achillea mellefolium extract,IN,ENG,N,
5,16818,acitretin,,IN,ENG,N,
6,1310220,Aconitum napellus extract,Aconitum napellus whole extract,IN,ENG,N,
7,272,activated charcoal,,IN,ENG,N,
8,281,acyclovir,,IN,ENG,N,
9,141400,adefovir dipivoxil,,PIN,ENG,N,


Save RxNorm Assets

In [15]:
rxcui_lookup_df.to_parquet(

    f"{OUTPUT_FOLDER}/rxcui_lookup.parquet",

    index=False
)

rxnorm_properties_df.to_parquet(

    f"{OUTPUT_FOLDER}/rxnorm_properties.parquet",

    index=False
)

print(
    "RxNorm assets saved"
)

RxNorm assets saved


Build Enriched Repository

In [16]:
drug_repository_enriched_df = (

    repository_df.merge(

        rxcui_lookup_df,

        on="generic_name",

        how="left"
    )

    .merge(

        rxnorm_properties_df,

        on="rxcui",

        how="left",

        suffixes=(
            "",
            "_rxnorm"
        )
    )
)

In [17]:
print(
    f"Rows: "
    f"{len(drug_repository_enriched_df):,}"
)

display(
    drug_repository_enriched_df.head(25)
)

Rows: 26,137


,product_ndc,brand_name,generic_name,manufacturer,dosage_form,route,marketing_category,application_number,product_type,package_ndc,package_description,sample,ingredient_name,strength,rxcui,name,synonym,tty,language,suppress,umlscui
0,59116-3393,None,Methylphenidate Hydrochloride,"Cambrex Charles City, Inc",POWDER,,BULK INGREDIENT,None,BULK INGREDIENT,59116-3393-5,60 kg in 1 DRUM (59116-3393-5),None,METHYLPHENIDATE HYDROCHLORIDE,1 kg/kg,203188,methylphenidate hydrochloride,,PIN,ENG,N,
1,59651-022,None,Loperamide Hydrochloride,Aurobindo Pharma Limited,TABLET,,DRUG FOR FURTHER PROCESSING,None,DRUG FOR FURTHER PROCESSING,59651-022-49,4000 TABLET in 1 BAG (59651-022-49),None,LOPERAMIDE HYDROCHLORIDE,2 mg/1,82038,loperamide hydrochloride,,PIN,ENG,N,
2,59651-860,None,Ubrogepant,Aurobindo Pharma Limited,POWDER,,BULK INGREDIENT,None,BULK INGREDIENT,59651-860-37,50 kg in 1 DRUM (59651-860-37),None,UBROGEPANT,50 kg/50kg,2268216,ubrogepant,,IN,ENG,N,
3,60312-0745,None,losartan potassium and hydrochlorothiazide,ORGANON PHARMA (UK) LIMITED,"TABLET, FILM COATED",,DRUG FOR FURTHER PROCESSING,None,DRUG FOR FURTHER PROCESSING,60312-0745-0,"1 BAG in 1 DRUM (60312-0745-0) / 72815 TABLET, FILM COATED in 1 BAG",None,HYDROCHLOROTHIAZIDE,12.5 mg/1,None,NaN,NaN,NaN,NaN,NaN,NaN
4,60312-0745,None,losartan potassium and hydrochlorothiazide,ORGANON PHARMA (UK) LIMITED,"TABLET, FILM COATED",,DRUG FOR FURTHER PROCESSING,None,DRUG FOR FURTHER PROCESSING,60312-0745-0,"1 BAG in 1 DRUM (60312-0745-0) / 72815 TABLET, FILM COATED in 1 BAG",None,LOSARTAN POTASSIUM,100 mg/1,None,NaN,NaN,NaN,NaN,NaN,NaN
5,60870-0288,None,Naloxone Hydrochloride,Aspen Oss B.V.,POWDER,,BULK INGREDIENT,None,BULK INGREDIENT,60870-0288-0,1 BAG in 1 DRUM (60870-0288-0) / 21000 g in 1 BAG,None,NALOXONE HYDROCHLORIDE,1 g/g,203192,naloxone hydrochloride,,PIN,ENG,N,
6,61662-0015,None,Degarelix Acetate,"Lianyungang Runzhong Pharmaceutical Co., Ltd.",POWDER,,BULK INGREDIENT,None,BULK INGREDIENT,61662-0015-1,4 kg in 1 BAG (61662-0015-1),None,DEGARELIX ACETATE,1 kg/kg,835863,degarelix acetate,,PIN,ENG,N,
7,62207-020,None,RIFAXIMIN,Granules India Limited,POWDER,,BULK INGREDIENT,None,BULK INGREDIENT,62207-020-02,25 kg in 1 DRUM (62207-020-02),None,RIFAXIMIN,1 kg/kg,35619,rifaximin,,IN,ENG,N,
8,62512-0024,None,CARBAMAZEPINE,Amoli Organics (A Division of Umedica Laboratories Pvt.Ltd.),POWDER,,BULK INGREDIENT,None,BULK INGREDIENT,62512-0024-1,50 kg in 1 DRUM (62512-0024-1),None,CARBAMAZEPINE,1 kg/kg,2002,carbamazepine,,IN,ENG,N,
9,62938-0004,None,Loratadine,CATALENT U.K. SWINDON ZYDIS LIMITED,"TABLET, ORALLY DISINTEGRATING",,DRUG FOR FURTHER PROCESSING,None,DRUG FOR FURTHER PROCESSING,62938-0004-1,"3024 BLISTER PACK in 1 BOX (62938-0004-1) / 10 TABLET, ORALLY DISINTEGRATING in 1 BLISTER PACK",None,LORATADINE,5 mg/1,28889,loratadine,,IN,ENG,N,


In [18]:
drug_repository_enriched_df.to_parquet(

    f"{OUTPUT_FOLDER}/drug_repository_enriched.parquet",

    index=False
)

print(
    "drug_repository_enriched.parquet saved"
)

drug_repository_enriched.parquet saved


Enrichment Profile

In [19]:
enrichment_statistics = {

    "repository_rows":
        int(
            len(drug_repository_enriched_df)
        ),

    "unique_products":
        int(
            drug_repository_enriched_df[
                "product_ndc"
            ].nunique()
        ),

    "unique_packages":
        int(
            drug_repository_enriched_df[
                "package_ndc"
            ].nunique()
        ),

    "unique_rxcui":
        int(
            rxnorm_properties_df[
                "rxcui"
            ].nunique()
        )
}

enrichment_statistics


{'repository_rows': 26137,
 'unique_products': 9915,
 'unique_packages': 18592,
 'unique_rxcui': 1457}

Production

In [20]:
with open(

    f"{OUTPUT_FOLDER}/rxnorm_enrichment_statistics.json",

    "w"
) as f:

    json.dump(
        enrichment_statistics,
        f,
        indent=4
    )

print(
    "rxnorm_enrichment_statistics.json saved"
)

rxnorm_enrichment_statistics.json saved


RxNorm Statistics

In [23]:
rxnorm_statistics = {

    "rxcui_lookup_rows":
        int(len(rxcui_lookup_df)),

    "unique_rxcui":
        int(
            rxnorm_properties_df[
                "rxcui"
            ].nunique()
        ),

    "rxnorm_property_rows":
        int(
            len(rxnorm_properties_df)
        ),

    "enriched_repository_rows":
        int(
            len(
                drug_repository_enriched_df
            )
        )
}

rxnorm_statistics


with open(

    f"{OUTPUT_FOLDER}/rxnorm_statistics.json",

    "w"
) as f:

    json.dump(
        rxnorm_statistics,
        f,
        indent=4
    )

print(
    "rxnorm_statistics.json saved"
)

rxnorm_statistics.json saved


RxNorm Metadata

In [24]:
from datetime import datetime

rxnorm_metadata = {

    "build_timestamp":

        datetime.now().strftime(
            "%Y-%m-%d %H:%M:%S"
        ),

    "unique_rxcui":

        int(
            rxnorm_properties_df[
                "rxcui"
            ].nunique()
        ),

    "repository_rows":

        int(
            len(
                drug_repository_enriched_df
            )
        )
}


with open(

    f"{OUTPUT_FOLDER}/rxnorm_metadata.json",

    "w"
) as f:

    json.dump(
        rxnorm_metadata,
        f,
        indent=4
    )

print(
    "rxnorm_metadata.json saved"
)

rxnorm_metadata.json saved


Build Data Dictionary

In [25]:
rxnorm_data_dictionary = pd.DataFrame({

    "column_name":

        drug_repository_enriched_df.columns,

    "data_type":

        drug_repository_enriched_df.dtypes.astype(str)
})


rxnorm_data_dictionary.to_csv(

    f"{OUTPUT_FOLDER}/rxnorm_data_dictionary.csv",

    index=False
)

print(
    "rxnorm_data_dictionary.csv saved"
)

rxnorm_data_dictionary.csv saved
